# 9.1 Meta Inference Platform Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/10_production_stories/09.1_meta_inference_platform/lab.ipynb)
[![Open In Molab](https://raw.githubusercontent.com/marimo-team/marimo/main/docs/_static/marimo-badge.svg)](https://molab.marimo.io/import/github/harshuljain13/llm-inference-at-scale/blob/master/content/10_production_stories/09.1_meta_inference_platform/lab.ipynb)

Analytical experiments exploring Meta's parallelism composition, network topology tradeoffs, and mixed-workload scheduling.

In [ ]:
# Install dependencies for analytical experiments
import subprocess
subprocess.run(['pip', 'install', '-q', 'matplotlib', 'numpy'], check=True)

import numpy as np  # numerical computation for parallelism models
import matplotlib.pyplot as plt  # plotting communication costs and utilization

## Experiment 1: Tensor Parallelism Communication Cost

Model the AllReduce overhead as TP degree increases within a node.

In [ ]:
def compute_tp_allreduce_cost():
    """Compute AllReduce latency for different TP degrees on NVLink."""
    # Hidden dimension for Llama 405B
    hidden_dim = 12288
    # FP16 = 2 bytes per element
    dtype_bytes = 2
    # NVLink bandwidth in bytes/second (900 GB/s bidirectional)
    nvlink_bw = 900e9
    # Number of transformer layers
    num_layers = 126
    # TP degrees to evaluate
    tp_degrees = np.array([1, 2, 4, 8])

    # AllReduce cost formula: 2 * (P-1)/P * M * dtype / bandwidth
    # where P = TP degree, M = message size (hidden_dim)
    per_layer_bytes = 2 * (tp_degrees - 1) / tp_degrees * hidden_dim * dtype_bytes
    # Time per layer in microseconds
    per_layer_us = per_layer_bytes / nvlink_bw * 1e6
    # Total overhead across all layers in milliseconds
    total_ms = per_layer_us * num_layers / 1000

    # Plot the communication overhead
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Left: per-layer cost
    ax1.bar(tp_degrees.astype(str), per_layer_us, color='#dbeafe', edgecolor='#000')
    ax1.set_xlabel('TP Degree')  # x-axis: parallelism width
    ax1.set_ylabel('Per-Layer AllReduce (\u00b5s)')  # y-axis: latency per layer
    ax1.set_title('AllReduce Cost per Transformer Layer')
    # Annotate each bar with its value
    for i, v in enumerate(per_layer_us):
        ax1.text(i, v + 0.001, f'{v:.3f}', ha='center', fontsize=9)

    # Right: total overhead across full model
    ax2.bar(tp_degrees.astype(str), total_ms, color='#dcfce7', edgecolor='#000')
    ax2.set_xlabel('TP Degree')  # x-axis: parallelism width
    ax2.set_ylabel('Total AllReduce Overhead (ms)')  # y-axis: total per-pass cost
    ax2.set_title(f'Total Overhead Across {num_layers} Layers (Llama 405B)')
    for i, v in enumerate(total_ms):
        ax2.text(i, v + 0.0001, f'{v:.4f}', ha='center', fontsize=9)

    plt.tight_layout()
    plt.show()

    # Return results for inspection
    return dict(zip(tp_degrees.tolist(), total_ms.tolist()))

# Execute the TP cost analysis
tp_results = compute_tp_allreduce_cost()
print(f"\nTP-8 AllReduce overhead: {tp_results[8]:.4f} ms per forward pass")
print("Conclusion: NVLink makes TP-8 viable with sub-millisecond overhead")

## Experiment 2: Context Parallelism Ring Bandwidth

Model the ring attention communication volume across RoCE v2 for different CP degrees and sequence lengths.

In [ ]:
def compute_cp_ring_cost():
    """Model ring attention data transfer for context parallelism."""
    # Parameters for Llama 405B
    head_dim = 128  # dimension per attention head
    batch_size = 32  # concurrent sequences in the batch
    # RoCE v2 bandwidth: 400 Gbps = 50 GB/s per link
    roce_bw = 50e9
    # Sequence lengths to evaluate
    seq_lengths = np.array([4096, 16384, 32768, 65536, 131072])
    # CP degrees to compare
    cp_degrees = [2, 4, 8]

    fig, ax = plt.subplots(figsize=(10, 5))
    # Color palette for each CP degree
    colors = ['#dbeafe', '#dcfce7', '#f3e8ff']

    for idx, cp in enumerate(cp_degrees):
        # Chunk length = total sequence / CP degree
        chunk_lens = seq_lengths / cp
        # Bytes per ring step: batch * chunk_len * head_dim * 2 (FP16)
        bytes_per_step = batch_size * chunk_lens * head_dim * 2
        # Time per ring step in milliseconds
        time_ms = bytes_per_step / roce_bw * 1000

        # Plot ring step latency for each sequence length
        ax.plot(seq_lengths / 1024, time_ms, 'o-',
                color=colors[idx], markeredgecolor='#000',
                linewidth=2, markersize=8, label=f'CP-{cp}')

    ax.set_xlabel('Sequence Length (K tokens)')  # x-axis: context size
    ax.set_ylabel('Ring Step Latency (ms)')  # y-axis: communication time
    ax.set_title('Context Parallelism Ring Communication (RoCE v2, 400 Gbps)')
    ax.legend()  # show CP degree legend
    ax.grid(True, alpha=0.3)  # light grid for readability
    # Add horizontal line for typical attention compute time
    ax.axhline(y=5.0, color='red', linestyle='--', alpha=0.5, label='Typical attn compute')
    ax.legend()
    plt.tight_layout()
    plt.show()

    # Print the critical case: 128K context with CP-4
    critical_bytes = batch_size * (131072 / 4) * head_dim * 2
    critical_ms = critical_bytes / roce_bw * 1000
    print(f"\n128K context, CP-4: {critical_ms:.2f} ms per ring step")
    print(f"This overlaps with attention compute (~5-10ms), hiding transfer cost")
    return critical_ms

# Execute CP bandwidth analysis
cp_critical_latency = compute_cp_ring_cost()

## Experiment 3: Mixed Workload GPU Utilization

Simulate how priority-based scheduling affects GPU utilization when mixing latency-sensitive and batch workloads.

In [ ]:
def simulate_mixed_workload_scheduling():
    """Simulate GPU utilization under mixed workload scheduling."""
    np.random.seed(42)  # reproducible results
    # Simulate 24 hours in 1-minute intervals
    hours = np.linspace(0, 24, 1440)

    # Chat traffic: peaks at 10am and 8pm, quiet at night
    # Modeled as sum of two Gaussians centered at peak hours
    chat_base = 0.3 * np.exp(-((hours - 10)**2) / 8) + 0.4 * np.exp(-((hours - 20)**2) / 6)
    # Add noise to simulate real traffic variability
    chat_load = np.clip(chat_base + np.random.normal(0, 0.03, len(hours)), 0, 1)

    # RLHF batch: fills remaining capacity (backfill workload)
    # Cannot exceed what's left after chat gets priority
    max_util = 0.85  # target fleet utilization cap
    rlhf_load = np.clip(max_util - chat_load, 0, 0.6)  # cap RLHF at 60%

    # Total utilization = chat + RLHF
    total_util = chat_load + rlhf_load

    # Plot the 24-hour utilization profile
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

    # Top: stacked area showing workload composition
    ax1.fill_between(hours, 0, chat_load, alpha=0.8, color='#dbeafe', label='Chat (P0)')
    ax1.fill_between(hours, chat_load, total_util, alpha=0.8, color='#dcfce7', label='RLHF (P2 backfill)')
    ax1.axhline(y=max_util, color='red', linestyle='--', alpha=0.7, label=f'Target: {max_util*100:.0f}%')
    ax1.set_ylabel('GPU Utilization')  # y-axis: fraction of GPU capacity used
    ax1.set_title('Mixed Workload Scheduling: Chat (Priority) + RLHF (Backfill)')
    ax1.legend(loc='upper right')  # workload type legend
    ax1.set_ylim(0, 1.0)  # utilization is 0-100%
    ax1.grid(True, alpha=0.2)

    # Bottom: comparison of dedicated vs shared utilization
    # Dedicated: chat gets its own GPUs (sized for peak), RLHF gets its own
    chat_dedicated_util = chat_load / chat_load.max()  # normalized to peak
    dedicated_avg = chat_load.mean() / chat_load.max()  # average util if dedicated
    shared_avg = total_util.mean()  # average util with sharing

    ax2.bar(['Dedicated\n(Chat only)', 'Shared Fleet\n(Chat + RLHF)'],
            [dedicated_avg * 100, shared_avg * 100],
            color=['#ffe4e6', '#dcfce7'], edgecolor='#000')
    ax2.set_ylabel('Avg GPU Utilization (%)')  # average across 24 hours
    ax2.set_title('Utilization: Dedicated vs Shared Fleet')
    # Annotate bars with values
    ax2.text(0, dedicated_avg * 100 + 1, f'{dedicated_avg*100:.1f}%', ha='center', fontsize=12)
    ax2.text(1, shared_avg * 100 + 1, f'{shared_avg*100:.1f}%', ha='center', fontsize=12)

    plt.xlabel('Hour of Day')  # x-axis for top plot
    plt.tight_layout()
    plt.show()

    print(f"\nDedicated fleet avg utilization: {dedicated_avg*100:.1f}%")
    print(f"Shared fleet avg utilization: {shared_avg*100:.1f}%")
    print(f"Improvement: {(shared_avg - dedicated_avg) / dedicated_avg * 100:.0f}% higher utilization")
    return {'dedicated': dedicated_avg, 'shared': shared_avg}

# Run the mixed workload simulation
workload_results = simulate_mixed_workload_scheduling()

## Experiment 4: Network Choice Impact on Parallelism Feasibility

Compare RoCE v2 vs InfiniBand for different parallelism dimensions.

In [ ]:
def compare_network_technologies():
    """Compare RoCE v2 and InfiniBand for inference parallelism."""
    # Network specs
    networks = {
        'NVLink (intra-node)': {'bw_gbps': 7200, 'latency_us': 0.5},  # 900 GB/s
        'InfiniBand NDR': {'bw_gbps': 400, 'latency_us': 1.0},
        'RoCE v2': {'bw_gbps': 400, 'latency_us': 2.5},
        'Ethernet (no RDMA)': {'bw_gbps': 400, 'latency_us': 15.0},
    }

    # Message sizes for different parallelism operations (bytes)
    operations = {
        'TP AllReduce\n(per layer, 405B)': 2 * 7/8 * 12288 * 2,  # ~21 KB
        'CP Ring Step\n(batch=32, 32K chunk)': 32 * 32768 * 128 * 2,  # ~256 MB
        'EP All-to-All\n(batch=32, top-2)': 32 * 2 * 12288 * 2,  # ~1.5 MB
    }

    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(operations))  # operation positions on x-axis
    width = 0.2  # bar width for grouped bars
    colors = ['#f3e8ff', '#dbeafe', '#dcfce7', '#ffe4e6']  # one per network

    for i, (net_name, specs) in enumerate(networks.items()):
        # Compute transfer time for each operation
        bw_bytes = specs['bw_gbps'] * 1e9 / 8  # convert Gbps to bytes/sec
        times_ms = []
        for op_name, msg_bytes in operations.items():
            # Total time = latency + transfer time
            transfer_ms = msg_bytes / bw_bytes * 1000
            total_ms = specs['latency_us'] / 1000 + transfer_ms
            times_ms.append(total_ms)

        # Plot grouped bars for this network
        bars = ax.bar(x + i * width, times_ms, width,
                      label=net_name, color=colors[i], edgecolor='#000')

    ax.set_xlabel('Parallelism Operation')  # x-axis: operation type
    ax.set_ylabel('Latency (ms, log scale)')  # y-axis: communication time
    ax.set_title('Communication Cost by Network Technology and Parallelism Type')
    ax.set_xticks(x + width * 1.5)  # center labels under groups
    ax.set_xticklabels(operations.keys())
    ax.set_yscale('log')  # log scale to show wide range
    ax.legend(loc='upper left')  # network type legend
    ax.grid(True, alpha=0.2, axis='y')

    plt.tight_layout()
    plt.show()

    # Print key insight
    print("\nKey insight: RoCE v2 vs InfiniBand difference is < 2 us latency")
    print("For CP ring steps (ms-scale), this is negligible")
    print("For TP AllReduce (us-scale), NVLink is essential regardless")

# Run network comparison
compare_network_technologies()